# Simulación de la llegada de los sismos
Se ejecuta **en Databricks**. Los acelerogramas de los 5 sismos ya están guardados en el Volume `fuente` (los subió `Subir_Datos_Databricks`). Este notebook los hace "llegar" uno por uno copiándolos a `landing/sismos/<evento>/`, que es donde los busca `01_bronze`.

Después de cada llegada se ejecuta el job con el rango de fechas del sismo (notebook `Crear_Workflow_Local`).

## 1. Rutas

In [ ]:
catalogo = "grupo06proyectofinal"
esquema = "sismos"
volumen = f"/Volumes/{catalogo}/{esquema}"
FUENTE = f"{volumen}/fuente/cismid"
LANDING = f"{volumen}/landing/sismos"
dbutils.fs.mkdirs(LANDING)

## 2. Funciones

In [ ]:
def sismos_en(carpeta):
    return sorted(e.name.rstrip("/") for e in dbutils.fs.ls(carpeta))


def simular_llegada(evento):
    dbutils.fs.cp(f"{FUENTE}/{evento}", f"{LANDING}/{evento}", recurse=True)
    print(f"Llegó el sismo {evento}: {len(dbutils.fs.ls(f'{LANDING}/{evento}'))} archivos en landing/sismos")


def estado():
    llegados = sismos_en(LANDING)
    for evento in sismos_en(FUENTE):
        print(f"  {evento}: {'llegó' if evento in llegados else 'pendiente'}")

## 3. Sismos guardados y sismos que ya llegaron

In [ ]:
estado()

## 4. Carga histórica: sismos 2007–2025
Luego se ejecuta el job con `fecha_desde = 2007-01-01` y `fecha_hasta = 2025-12-31` (notebook `Crear_Workflow_Local`).

In [ ]:
for evento in ["2007-08-15_pisco_Mw7.9", "2021-11-28_callao_O_M5.2", "2022-01-07_lima_NE_M5.6", "2025-06-15_callao_M6.1"]:
    simular_llegada(evento)
estado()

## 5. Llega un sismo nuevo: Ica, 19 de mayo de 2026
Luego se ejecuta el job con `fecha_desde = 2026-01-01` y `fecha_hasta = 2026-12-31`.

In [ ]:
simular_llegada("2026-05-19_ica_M6.1")
estado()